--- Day 8: Playground ---
Equipped with a new understanding of teleporter maintenance, you confidently step onto the repaired teleporter pad.

You rematerialize on an unfamiliar teleporter pad and find yourself in a vast underground space which contains a giant playground!

Across the playground, a group of Elves are working on setting up an ambitious Christmas decoration project. Through careful rigging, they have suspended a large number of small electrical junction boxes.

Their plan is to connect the junction boxes with long strings of lights. Most of the junction boxes don't provide electricity; however, when two junction boxes are connected by a string of lights, electricity can pass between those two junction boxes.

The Elves are trying to figure out which junction boxes to connect so that electricity can reach every junction box. They even have a list of all of the junction boxes' positions in 3D space (your puzzle input).

For example:

162,817,812
57,618,57
906,360,560
592,479,940
352,342,300
466,668,158
542,29,236
431,825,988
739,650,466
52,470,668
216,146,977
819,987,18
117,168,530
805,96,715
346,949,466
970,615,88
941,993,340
862,61,35
984,92,344
425,690,689
This list describes the position of 20 junction boxes, one per line. Each position is given as X,Y,Z coordinates. So, the first junction box in the list is at X=162, Y=817, Z=812.

To save on string lights, the Elves would like to focus on connecting pairs of junction boxes that are as close together as possible according to straight-line distance. In this example, the two junction boxes which are closest together are 162,817,812 and 425,690,689.

By connecting these two junction boxes together, because electricity can flow between them, they become part of the same circuit. After connecting them, there is a single circuit which contains two junction boxes, and the remaining 18 junction boxes remain in their own individual circuits.

Now, the two junction boxes which are closest together but aren't already directly connected are 162,817,812 and 431,825,988. After connecting them, since 162,817,812 is already connected to another junction box, there is now a single circuit which contains three junction boxes and an additional 17 circuits which contain one junction box each.

The next two junction boxes to connect are 906,360,560 and 805,96,715. After connecting them, there is a circuit containing 3 junction boxes, a circuit containing 2 junction boxes, and 15 circuits which contain one junction box each.

The next two junction boxes are 431,825,988 and 425,690,689. Because these two junction boxes were already in the same circuit, nothing happens!

This process continues for a while, and the Elves are concerned that they don't have enough extension cables for all these circuits. They would like to know how big the circuits will be.

After making the ten shortest connections, there are 11 circuits: one circuit which contains 5 junction boxes, one circuit which contains 4 junction boxes, two circuits which contain 2 junction boxes each, and seven circuits which each contain a single junction box. Multiplying together the sizes of the three largest circuits (5, 4, and one of the circuits of size 2) produces 40.

Your list contains many junction boxes; connect together the 1000 pairs of junction boxes which are closest together. Afterward, what do you get if you multiply together the sizes of the three largest circuits?

Notes:
This is a multi part puzzle.

We need to
- find the 1000 shortest connections and connect them
- identify circuits within those connections (common nodes)
- find the 3 largest circuits and multiply their sizes


For finding the shortest connections, we can try just brute-forcing it (its around a million total pairs). It could be easier to cluster first and then only evaluate for adjacent clusters. But let's try doing it clunky 

In [31]:
with open('input.txt') as file:
    boxes = tuple(tuple(int(coordinate) for coordinate in box.rstrip().split(',')) for box in file)

def dist(box_a, box_b): # we can leave all distances squared since only ordering matters
    return (box_a[0] - box_b[0])**2 + (box_a[1] - box_b[1])**2 + (box_a[2] - box_b[2])**2

# get rough upper bound for distances in dataset:
import random
sum_dist = 0
for _ in range(10000):
    sum_dist += dist(random.choice(boxes), random.choice(boxes))
print(sum_dist / 10000)

count = 0
for _ in range(10000):
    if dist(random.choice(boxes), random.choice(boxes)) < sum_dist / 10000 / 50:
        count += 1
print(count)
# ok, 1% of avg is decent cutoff to populate shortlist
cutoff = sum_dist / 10000 / 50

5001633310.5973
48


In [34]:
shortlist = []
for i, box_a in enumerate(boxes):
    for box_b in boxes[i+1:]:
        if dist(box_a, box_b) < cutoff:
            shortlist.append((box_a, box_b, dist(box_a, box_b)))
len(shortlist)

1913

In [38]:
shortlist.sort(key = lambda x: x[2])
closest_1000 = shortlist[:1000]

In [56]:
circuits = []
for box_a, box_b, _ in closest_1000:
    mod_circuits = []
    for circuit in circuits:
        """
        cases:
        - if Box A is in a set, add Box B and vice versa
        - if Box A and Box B are already in different sets, merge the sets
        - if neither Box A nor Box B are in any set, add both of them to a new circuit"""
        if box_a in circuit or box_b in circuit:
            circuit.add(box_a)
            circuit.add(box_b)
            circuits.remove(circuit)
            mod_circuits.append(circuit)
    if not mod_circuits:
        circuits.append({box_a, box_b})
    elif len(mod_circuits) > 1:
        combined_circuit = set.union(*mod_circuits)
        circuits.append(combined_circuit)
    elif len(mod_circuits) == 1:
        circuits.append(mod_circuits[0])

In [60]:
circuits.sort(key = lambda x: len(x), reverse=True)
print(len(circuits[0]) * len(circuits[1]) * len(circuits[2]))

153328


Your puzzle answer was 153328.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
The Elves were right; they definitely don't have enough extension cables. You'll need to keep connecting junction boxes together until they're all in one large circuit.

Continuing the above example, the first connection which causes all of the junction boxes to form a single circuit is between the junction boxes at 216,146,977 and 117,168,530. The Elves need to know how far those junction boxes are from the wall so they can pick the right extension cable; multiplying the X coordinates of those two junction boxes (216 and 117) produces 25272.

Continue connecting the closest unconnected pairs of junction boxes together until they're all in the same circuit. What do you get if you multiply together the X coordinates of the last two junction boxes you need to connect?

Ok this will be a bit more involved. At each step we need to:
- Find the shortest remaining connection among boxes that have not been connected (are not in the same circuit)
- update our list of circuits
- Check if (1) all boxes have been used at least once and (2) there is only 1 circuit left

In [64]:
# let's see if we can make one giant look up table of all connections sorted by distance:
longlist = []
for i, box_a in enumerate(boxes):
    for box_b in boxes[i+1:]:
        longlist.append((box_a, box_b, dist(box_a, box_b)))
longlist.sort(key=lambda x: x[2])
len(longlist)

499500

In [67]:
# see when all boxes have been used at least once:
last_box = 0
for box in boxes:
    for i, pair in enumerate(longlist):
        if box in pair:
            if i > last_box:
                last_box = i
            break
last_box

5243

In [97]:
circuits = []
for num_connections, (box_a, box_b, _) in enumerate(longlist):
    mod_circuits = []
    for c in circuits:
        if box_a in c or box_b in c:
            c.add(box_a)
            c.add(box_b)
            mod_circuits.append(c)
    if not mod_circuits:
        circuits.append({box_a, box_b})
    elif len(mod_circuits) > 1:
        for c in mod_circuits:
            circuits.remove(c)
        combined_circuit = set.union(*mod_circuits)
        circuits.append(combined_circuit)
    if num_connections >= last_box and len(circuits) == 1:
        print(f'The last two boxes connected where: {box_a, box_b}')
        print(f'Their product of X-coordinates is {box_a[0] * box_b[0]=}')
        print(f'{num_connections} connections were added')
        break


The last two boxes connected where: ((71430, 41887, 1462), (85337, 44242, 3891))
Their product of X-coordinates is box_a[0] * box_b[0]=6095621910
5243 connections were added


That's the right answer! You are one gold star closer to decorating the North Pole.